In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os 

Dataset = '/kaggle/input/competitions/google-research-identify-contrails-reduce-global-warming'

train = os.listdir(Dataset+"/train")
print(f"Nombre d'exemples dans train :  {len(train)}")
print("Quelques record_id :", train[:5])


### Test un example 

In [ ]:
test = train[0]
test_dir = Dataset+'/train/'+test

print("Fichier contenu :",os.listdir(test_dir))


band8 = np.load(test_dir+'/band_08.npy')
mask = np.load(test_dir+'/human_pixel_masks.npy')

print("Shape band_08 :", band8.shape)
print("Shape masque  :", mask.shape)
print("Min/Max band_08 :", band8.min(), band8.max())
print("Nb pixels positifs dans le masque :", mask.sum())

In [ ]:
test

In [ ]:
N_TIMES_BEFORE = 4

def get_target_frame(arr, time_axis_last=True, idx=N_TIMES_BEFORE):
    """Extrait la frame 2D correspondant à l'image labellisée."""
    return arr[..., idx] if time_axis_last else arr[idx]

frame = get_target_frame(band8)
plt.figure(figsize=(5, 5))
plt.imshow(frame, cmap='gray')
plt.title(f"band_08 — {test}")
plt.colorbar(label='Température de brillance (K)')
plt.show()

In [ ]:
positive_id = None
for rid in train[:300]:
    m = np.load(os.path.join(Dataset+'/train', rid, 'human_pixel_masks.npy'))
    if m.sum() > 0:
        positive_id = rid
        break

print("Exemple positif trouvé :", positive_id)

In [ ]:
example_dir = os.path.join(Dataset+'/train', positive_id)
band14 = np.load(os.path.join(test_dir, 'band_14.npy'))
mask = np.load(os.path.join(test_dir, 'human_pixel_masks.npy'))

frame = get_target_frame(band14)
mask_2d = mask.squeeze()  # enlève les dimensions de taille 1

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(frame, cmap='gray')
axes[0].set_title('band_14 (11µm)')
axes[1].imshow(mask_2d, cmap='gray')
axes[1].set_title('Masque contrail (vérité terrain)')
plt.show()

In [ ]:
def load_band(d, name):
    return np.load(os.path.join(d, f'{name}.npy'))

def get_ash_composite(example_dir, idx=N_TIMES_BEFORE):
    b11 = get_target_frame(load_band(test_dir, 'band_11'), idx=idx)  # ~8µm
    b14 = get_target_frame(load_band(test_dir, 'band_14'), idx=idx)  # ~11µm
    b15 = get_target_frame(load_band(test_dir, 'band_15'), idx=idx)  # ~12µm

    r = b15
    g = b14 - b11
    b = b15 - b14

    def normalize(x):
        return np.clip((x - x.min()) / (x.max() - x.min() + 1e-6), 0, 1)

    return np.stack([normalize(r), normalize(g), normalize(b)], axis=-1)

ash = get_ash_composite(example_dir)
plt.figure(figsize=(6, 6))
plt.imshow(ash)
plt.title(f"Composite ash — {positive_id}")
plt.show()

Distribution des données


In [ ]:
N_SAMPLE = 2000
samples = train[:N_SAMPLE]

positive_fractions = []
has_contrail = []

for rid in samples:
    mask = np.load(os.path.join(Dataset + '/train', rid, 'human_pixel_masks.npy'))
    positive_fractions.append(mask.sum() / mask.size)
    has_contrail.append(mask.sum() > 0)

positive_fractions = np.array(positive_fractions)
has_contrail = np.array(has_contrail)

In [ ]:
n_neg = (~has_contrail).sum()
n_pos = has_contrail.sum()

plt.figure(figsize=(5, 4))
plt.bar(['Sans contrail', 'Avec contrail'], [n_neg, n_pos], color=['gray', 'steelblue'])
plt.ylabel("Nombre d'exemples")
plt.title(f"Répartition sur {N_SAMPLE} exemples")
plt.show()

print(f"{n_pos}/{N_SAMPLE} exemples positifs ({100*n_pos/N_SAMPLE:.1f}%)")

In [ ]:
positive_only_pct = positive_fractions[has_contrail] * 100

plt.figure(figsize=(6, 4))
plt.hist(positive_only_pct, bins=40, color='steelblue', edgecolor='white')
plt.xlabel('% de pixels contrail dans l\'image')
plt.ylabel("Nombre d'exemples")
plt.title("Distribution du taux de pixels positifs (exemples positifs seulement)")
plt.show()

print(f"Taux médian : {np.median(positive_only_pct):.3f}%")
print(f"Taux max observé : {positive_only_pct.max():.3f}%")

In [ ]:
N_TIMES_BEFORE = 4
N_SUBSET = 100

positive_ids = [rid for rid, has in zip(sample_ids, has_contrail) if has][:N_SUBSET]

contrail_vals, background_vals = [], []

for rid in positive_ids:
    d = os.path.join(Dataset + '/train', rid)
    frame = np.load(os.path.join(d, 'band_14.npy'))[:, :, N_TIMES_BEFORE]
    mask_2d = np.load(os.path.join(d, 'human_pixel_masks.npy')).squeeze().astype(bool)
    contrail_vals.append(frame[mask_2d])
    background_vals.append(frame[~mask_2d])

contrail_vals = np.concatenate(contrail_vals)
background_vals = np.concatenate(background_vals)

plt.figure(figsize=(7, 5))
plt.hist(background_vals, bins=60, alpha=0.5, density=True, label='Fond', color='gray')
plt.hist(contrail_vals, bins=60, alpha=0.6, density=True, label='Contrail', color='crimson')
plt.xlabel('band_14 — Température de brillance (K)')
plt.ylabel('Densité')
plt.legend()
plt.title('Contrail vs fond : bande brute')
plt.show()

print(f"Moyenne fond     : {background_vals.mean():.1f} K")
print(f"Moyenne contrail : {contrail_vals.mean():.1f} K")

In [ ]:
contrail_diff, background_diff = [], []

for rid in positive_ids:
    d = os.path.join(Dataset + '/train', rid)
    b14 = np.load(os.path.join(d, 'band_14.npy'))[:, :, N_TIMES_BEFORE]
    b15 = np.load(os.path.join(d, 'band_15.npy'))[:, :, N_TIMES_BEFORE]
    diff = b15 - b14
    mask_2d = np.load(os.path.join(d, 'human_pixel_masks.npy')).squeeze().astype(bool)
    contrail_diff.append(diff[mask_2d])
    background_diff.append(diff[~mask_2d])

contrail_diff = np.concatenate(contrail_diff)
background_diff = np.concatenate(background_diff)

plt.figure(figsize=(7, 5))
plt.hist(background_diff, bins=60, alpha=0.5, density=True, label='Fond', color='gray')
plt.hist(contrail_diff, bins=60, alpha=0.6, density=True, label='Contrail', color='crimson')
plt.xlabel('band_15 - band_14 (K)')
plt.ylabel('Densité')
plt.legend()
plt.title('Contrail vs fond : canal différence')
plt.show()